# 项目 1：TESS 系外行星凌星 — WASP-12b

流程：`tess_lightcurve` 获取数据（io 层，缓存/入库/重试全自动）→ 目录周期折叠 → TLS 盲搜周期 → 与目录值对比。

数据获取走 myastro.io：首次运行下载，此后断网也能跑（缓存命中）。

In [ ]:
from myastro.io.sources.tess import tess_lightcurve
from myastro.io.sources.exoplanet import exoplanet_params

result = tess_lightcurve("WASP-12", sector=20, cadence="2m")
print("cache_hit:", result.cache_hit, "| meta:", result.meta)
lc_path = result.paths[0]

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from myastro import lc
plt.rcParams["font.sans-serif"] = ["PingFang SC", "Arial Unicode MS", "Heiti TC"]

df = pd.read_parquet(lc_path)
df = lc.quality_cut(df)                  # 去 bad cadence（quality != 0）
t = df.time.values
f = lc.normalize(df.flux.values)         # 归一化（TLS 要求，库内也会强制）
print(f"{len(t)} cadences, {t.max()-t.min():.1f} 天基线")


## 1. 目录周期折叠

In [ ]:
cat = exoplanet_params("WASP-12 b")
P_cat = cat["pl_orbper"]; T0 = cat["pl_tranmid"]
print(f"目录: P = {P_cat:.6f} ± {cat['pl_orbpererr1']:.6f} d, "
      f"T0 = {T0:.3f} (BJD), depth = {cat['pl_trandep']*1e4:.0f} ppm")

In [ ]:
import matplotlib.pyplot as plt

# 目录周期折叠：t0=T0（凌星居中，相位 0 = 凌星中点）；fold 返回值已按相位排序
phase, f_sorted = lc.fold(t, f, P_cat, t0=T0)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(phase, f_sorted, ".", ms=1, alpha=0.3)
ax.set(xlabel="相位 (P_catalog)", ylabel="PDCSAP flux",
       title=f"WASP-12b TESS S20 折叠 (P={P_cat:.5f} d)")
ax.axvline(0, color="r", ls=":", lw=0.8)
fig.tight_layout(); fig.savefig("out/01-fold-catalog.png", dpi=150); fig


凌星在相位 0 处呈 V 形凹陷（WASP-12b 是热木星，深度大、时长短）。

## 2. TLS 盲搜周期（不用目录值）

In [ ]:
# 离群剔除（凌星深度仅 ~1.4%，0.9–1.1 之外必为异常）→ 库内 TLS 盲搜
tt, ff = lc.clip_range(f, 0.9, 1.1, t=t)
print(f"离群剔除: {len(f)-len(ff)}/{len(f)} 点")

# 网格限定 0.5–3 d：热木星先验。transit_search 内部强制归一化，
# 并返回真深度（TLS 原生 depth 是凌内/凌外比，两处坑已在库内焊死）
res = lc.transit_search(tt, ff, period_min=0.5, period_max=3.0)
print(f"TLS: P = {res.period:.6f} d, depth = {res.depth*1e6:.0f} ppm, "
      f"duration = {res.duration_days*24*60:.1f} min, SNR = {res.snr:.1f}")


In [ ]:
rel = abs(res.period - P_cat) / P_cat
print(f"盲搜 vs 目录: TLS {res.period:.6f} d vs 目录 {P_cat:.6f} d，"
      f"相对偏差 {rel*100:.4f}%")
assert rel < 0.01, "偏差超过 1%，检查数据或参数"

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(res.raw.periods, res.raw.power, lw=0.5)
ax.axvline(res.period, color="r", ls="--", label=f"TLS {res.period:.5f} d")
ax.axvline(P_cat, color="g", ls=":", label=f"目录 {P_cat:.5f} d")
ax.set(xlabel="周期 (d)", ylabel="TLS power", title="TLS 周期图（网格 0.5–3 d）")
ax.legend(); fig.tight_layout(); fig.savefig("out/01-tls-periodogram.png", dpi=150); fig


## 3. 结论

- TLS 盲搜周期 1.090964 d，与目录 1.091420 d 偏差 **0.042%**；深度 ~1.57%（目录 1.39%，TESS 大像元稀释所致），时长 138 min，SNR 272——检测可信
- 两个坑已焊进 `myastro.lc.transit_search`（不再是文字教训）：PDCSAP 通量必须归一化再进 TLS（否则 no transit fit 静默返回 nan）；TLS 的 `depth` 是凌内/凌外通量比，真深度 = `1 - depth`
- 目录多解选择已确定性化（`exoplanet_params` 取"周期与深度完整且周期误差最小"的解）——复跑时曾因行序漂移拿到 `pl_trandep=None`
- 全程数据与参数入库（astro.db），二次运行零网络
